# Sanday — Colab Training

Run cells top-to-bottom. Requires a Colab GPU runtime (T4 or better).

## 1. Repo

In [13]:
from pathlib import Path
import os, subprocess, sys

REPO_URL   = 'https://github.com/karl4th/sanday.git'
COLAB_REPO = Path('/content/sanday')

def _is_repo(path: Path) -> bool:
    return all((path / d).is_dir() for d in ('configs', 'scripts', 'sanday'))

def find_repo_root() -> Path:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if _is_repo(candidate):
            out = subprocess.run(
                ['git', '-C', str(candidate), 'pull', '--ff-only'],
                capture_output=True, text=True
            )
            print(out.stdout.strip() or 'Already up to date.')
            return candidate
    if COLAB_REPO.exists():
        subprocess.run(['git', '-C', str(COLAB_REPO), 'pull', '--ff-only'], check=True)
    else:
        subprocess.run(['git', 'clone', REPO_URL, str(COLAB_REPO)], check=True)
    return COLAB_REPO.resolve()

REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print('Repo root:', REPO_ROOT)

Updating 2424641..c2df4ba
Fast-forward
 configs/sanday_prepared_10k.yaml        |   3 +-
 notebooks/sanday_colab_experiment.ipynb | 295 ++++++++++++++++++++++++++++++--
 sanday/data.py                          |  16 ++
 scripts/evaluate.py                     |   1 +
 scripts/train.py                        |   2 +
 5 files changed, 298 insertions(+), 19 deletions(-)
Repo root: /content/sanday


## 2. Dependencies

In [2]:
%pip install -q -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 59.9 MB/s eta 0:00:0000:01


## 3. Dataset

Mounts Google Drive as a persistent cache — the 15 GB archive is downloaded once and reused across sessions.
If Drive is unavailable, falls back to the in-session Colab cache.

In [3]:
import subprocess
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive', force_remount=False)

DATASET_ROOT = Path('/content/dataset')
ARCHIVE_DIR = Path('/content/drive/MyDrive/sanday_datasets')
archives = sorted(ARCHIVE_DIR.glob('common_voice_en_wav_*.tar'), key=lambda p: p.stat().st_mtime)
assert archives, f'No dataset archive found in {ARCHIVE_DIR}'
DATASET_ARCHIVE = archives[-1]
print('Dataset archive:', DATASET_ARCHIVE)

if not (DATASET_ROOT / 'metadata.csv').exists() or not (DATASET_ROOT / 'wav').exists():
    print('Extracting dataset to /content ...')
    subprocess.run(['tar', '-xf', str(DATASET_ARCHIVE), '-C', '/content'], check=True)
else:
    print('Dataset already extracted:', DATASET_ROOT)

metadata_path = DATASET_ROOT / 'metadata.csv'
wav_dir = DATASET_ROOT / 'wav'
assert metadata_path.exists(), f'Missing metadata: {metadata_path}'
assert wav_dir.exists(), f'Missing wav dir: {wav_dir}'

wav_count = sum(1 for _ in wav_dir.glob('*.wav'))
print('Dataset root:', DATASET_ROOT)
print('WAV files:', wav_count)
print('Dataset size:')
subprocess.run(['du', '-sh', str(DATASET_ROOT)], check=False)
print('Metadata preview:')
subprocess.run(['head', '-5', str(metadata_path)], check=False)


Mounted at /content/drive
Dataset archive: /content/drive/MyDrive/sanday_datasets/common_voice_en_wav_21748_20260621_112829.tar
Extracting dataset to /content ...
Dataset root: /content/dataset
WAV files: 21748
Dataset size:
Metadata preview:


CompletedProcess(args=['head', '-5', '/content/dataset/metadata.csv'], returncode=0)

## 4. Environment

In [4]:
import torch

assert torch.cuda.is_available(), 'No GPU — switch runtime to GPU in Colab'
print('GPU:  ', torch.cuda.get_device_name(0))
print('VRAM: ', f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print('Torch:', torch.__version__)

GPU:   Tesla T4
VRAM:  15.6 GB
Torch: 2.11.0+cu128


## 5. Configuration

In [14]:
CONFIG     = 'configs/sanday_prepared_10k.yaml'
RUN_NAME   = 'prepared_10k_seed42'
RUN_DIR    = f'results/{RUN_NAME}'
SMOKE_DIR  = f'{RUN_DIR}_smoke'
EVAL_DIR   = f'{RUN_DIR}_eval'
CHECKPOINT = f'{RUN_DIR}/checkpoints/sanday_best.pt'

RUN_SMOKE_TEST = True
TRAIN          = True
EVALUATE       = True

print('Config:    ', CONFIG)
print('Run dir:   ', RUN_DIR)
print('Checkpoint:', CHECKPOINT)


Config:     configs/sanday_prepared_10k.yaml
Run dir:    results/prepared_10k_seed42
Checkpoint: results/prepared_10k_seed42/checkpoints/sanday_best.pt


## 6. Helper

In [6]:
import shlex, subprocess, sys
from collections import deque

def run_command(cmd, run_dir=None):
    print('Running:', ' '.join(shlex.quote(str(p)) for p in cmd), flush=True)
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    tail = deque(maxlen=300)
    for line in proc.stdout:
        print(line, end='')
        tail.append(line)
    rc = proc.wait()
    if rc != 0:
        print(f'\nFailed (exit {rc})')
        log = Path(run_dir) / 'error.log' if run_dir else None
        if log and log.exists():
            print('\n--- error.log ---\n', log.read_text()[-8000:])
        else:
            print('\n--- output tail ---\n', ''.join(tail)[-8000:])
        raise RuntimeError('Command failed')
    return rc

## 7. Smoke Test

2 train batches + 1 valid batch with a tiny model. Should finish in ~15 seconds.

In [15]:
if RUN_SMOKE_TEST:
    run_command([
        sys.executable, 'scripts/train.py',
        '--config', CONFIG,
        '--run-dir', SMOKE_DIR,
        '--epochs', '1',
        '--max-items', '100',
        '--max-train-batches', '2',
        '--max-valid-batches', '1',
    ], SMOKE_DIR)
else:
    print('Skipped.')


Running: /usr/bin/python3 scripts/train.py --config configs/sanday_prepared_10k.yaml --run-dir results/prepared_10k_seed42_smoke --epochs 1 --max-items 100 --max-train-batches 2 --max-valid-batches 1
Dataset root: /content/dataset
Run dir: results/prepared_10k_seed42_smoke
Seed: 42
Trainable parameters: 8,248,381
Total parameters: 8,971,981

Training failed. Full traceback written to: results/prepared_10k_seed42_smoke/error.log
Traceback (most recent call last):
  File "/content/sanday/scripts/train.py", line 365, in <module>
    main()
  File "/content/sanday/scripts/train.py", line 147, in main
    train_dataset = CommonVoiceDataset(
                    ^^^^^^^^^^^^^^^^^^^
  File "/content/sanday/sanday/data.py", line 45, in __init__
    self.table = self._filter_by_duration(max_audio_seconds).reset_index(drop=True)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/sanday/sanday/data.py", line 62, in _filter_by_duration
    info = torchaudio.info(audio_pat

RuntimeError: Command failed

## 8. Train

In [12]:
if TRAIN:
    run_command([
        sys.executable, 'scripts/train.py',
        '--config', CONFIG,
        '--run-dir', RUN_DIR,
    ], RUN_DIR)
else:
    print('Skipped.')

Running: /usr/bin/python3 scripts/train.py --config configs/sanday_prepared_10k.yaml --run-dir results/prepared_10k_seed42
Dataset root: /content/dataset
Run dir: results/prepared_10k_seed42
Seed: 42
Trainable parameters: 8,248,381
Total parameters: 8,971,981
CommonVoiceDataset split=train manifest=/content/dataset/metadata.csv rows=8000
CommonVoiceDataset split=valid manifest=/content/dataset/metadata.csv rows=1000

Training failed. Full traceback written to: results/prepared_10k_seed42/error.log
Traceback (most recent call last):
  File "/content/sanday/scripts/train.py", line 363, in <module>
    main()
  File "/content/sanday/scripts/train.py", line 246, in main
    logits, output_lengths = model(mel, input_lengths)
                             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1779, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/loca

RuntimeError: Command failed

## 9. Training Results

In [ ]:
import json
import pandas as pd
from IPython.display import display

summary_path = Path(RUN_DIR) / 'summary.json'
metrics_path = Path(RUN_DIR) / 'metrics.csv'

if summary_path.exists():
    print(json.dumps(json.loads(summary_path.read_text()), indent=2))
else:
    print('No summary yet.')

if metrics_path.exists():
    df = pd.read_csv(metrics_path)
    display(df.tail(10))
    display(df[['epoch', 'train_loss', 'valid_wer', 'valid_cer']].plot(
        x='epoch', grid=True, figsize=(10, 4)
    ))
else:
    print('No metrics yet.')

## 10. Evaluate

In [ ]:
if EVALUATE:
    ckpt = Path(CHECKPOINT)
    if not ckpt.exists():
        raise FileNotFoundError(f'Checkpoint not found: {ckpt}')
    run_command([
        sys.executable, 'scripts/evaluate.py',
        '--config', CONFIG,
        '--checkpoint', str(ckpt),
        '--run-dir', EVAL_DIR,
    ], EVAL_DIR)
else:
    print('Skipped.')

## 11. Evaluation Results

In [ ]:
eval_path = Path(EVAL_DIR) / 'evaluation.json'
pred_path = Path(EVAL_DIR) / 'predictions.csv'

if eval_path.exists():
    print(json.dumps(json.loads(eval_path.read_text()), indent=2))
else:
    print('No evaluation yet.')

if pred_path.exists():
    display(pd.read_csv(pred_path).head(20))
else:
    print('No predictions yet.')

## 12. Artifacts

In [ ]:
for p in [
    Path(RUN_DIR) / 'config.yaml',
    Path(RUN_DIR) / 'metrics.csv',
    Path(RUN_DIR) / 'summary.json',
    Path(EVAL_DIR) / 'evaluation.json',
    Path(EVAL_DIR) / 'predictions.csv',
]:
    print('OK  ' if p.exists() else 'MISS', p)
print('\nCheckpoint (large, excluded from git):', Path(CHECKPOINT))